# Invoice extraction: from rules to trained models

This notebook runs the whole pipeline step by step and shows the results:

1. **Ground truth without manual labelling** — payment QR codes and archive file names
2. **Benchmark** — today's rule-based extractor, alone and with Gemma 4 filling the gaps
3. **Supplier split** — train / validation / test with no supplier in two sets
4. **Own OCR** — a CRNN text recognizer trained from scratch
5. **Own extraction model** — LiLT, a layout-aware transformer, fine-tuned on QR-labelled invoices
6. **Headline numbers**

**How to use it.** Select the kernel `.venv-train`, then *Run All*. Long steps
are switched off in `RUN` below, so Run All only shows results that already
exist; set a step to `True` to (re)compute it. Steps that need the app's OCR
run in the app's environment (`.venv`), training steps in `.venv-train`.

**Privacy.** Invoices stay in gitignored folders (`eval_data/`, `eval_results/`,
`training/data/`, `training/runs/`). Every cell shows aggregates only — counts,
rates, timings, training curves — never invoice contents.

In [ ]:
RUN = {
    "labels": False,      # 1. QR + file-name ground truth                      ~5 min
    "benchmark": False,   # 2. rules vs rules + Gemma 4 on all documents        ~60-90 min, GPU
    "split": False,       # 3. supplier-grouped split                           seconds
    "real_lines": False,  # 4a. real text lines from digital invoices           ~5 min
    "train_ocr": False,   # 4b. train the CRNN                                  ~1-2 h, GPU
    "test_ocr": False,    # 4c. CRNN on test-split lines                        ~1 min
    "dump_words": False,  # 5a. words + positions (OCR for scans)              ~30-40 min, GPU
    "train_lilt": False,  # 5b. fine-tune LiLT                                  ~20-30 min, GPU
    "test_lilt": False,   # 5c. LiLT vs rules on the test split                 ~2 min
}

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "extractor.py").exists())
sys.path.insert(0, str(ROOT))
from evaluation.metrics import FIELD_LABELS  # noqa: E402

EXE = Path("Scripts", "python.exe") if os.name == "nt" else Path("bin", "python")
APP_PY = ROOT / ".venv" / EXE  # the app's environment: PaddleOCR, Ollama client
TRAIN_PY = ROOT / ".venv-train" / EXE  # PyTorch + Transformers

DOCS = ROOT / "eval_data" / "docs"
LABELS = ROOT / "eval_data" / "labels_qr.csv"
BENCHMARK = ROOT / "eval_results" / "qr_labels_gemma4"
SPLIT = ROOT / "training" / "data" / "split.csv"
REAL_LINES = ROOT / "training" / "data" / "real_lines"
WORDS = ROOT / "training" / "data" / "words"
OCR_RUN = ROOT / "training" / "runs" / "ocr" / "crnn"
LILT_RUN = ROOT / "training" / "runs" / "lilt" / "v1"
LLM = "gemma4:e4b-it-qat"
PCT = "{:.1%}"


def run(python, *args):
    """Run one pipeline step as a module in the given environment, streaming its progress."""
    cmd = [str(python), "-m", *map(str, args)]
    print(">", " ".join(cmd[2:]), flush=True)
    env = {**os.environ, "PYTHONIOENCODING": "utf-8"}
    with subprocess.Popen(cmd, cwd=ROOT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace") as process:
        for line in process.stdout:
            if "Redirects are currently not supported" not in line:
                print(line, end="", flush=True)
    if process.returncode:
        raise RuntimeError(f"step failed with exit code {process.returncode}")


def load(path):
    path = Path(path)
    if not path.exists():
        print(f"not produced yet: {path.relative_to(ROOT)}")
        return None
    return json.loads(path.read_text(encoding="utf-8"))


def read_csv(path):
    path = Path(path)
    if not path.exists():
        print(f"not produced yet: {path.relative_to(ROOT)}")
        return None
    return pd.read_csv(path, encoding="utf-8-sig", dtype=str).fillna("")


def rates(correct, wrong, missing):
    total, filled = correct + wrong + missing, correct + wrong
    accuracy = correct / total if total else None
    precision = correct / filled if filled else None
    if precision is None:
        return accuracy, precision, None
    return accuracy, precision, (2 * precision * accuracy / (precision + accuracy) if precision + accuracy else 0.0)


def score_rows(name, scores):
    rows = []
    for field, s in scores.items():
        total = s["correct"] + s["wrong"] + s["missing"]
        if total:
            acc, prec, f1 = rates(s["correct"], s["wrong"], s["missing"])
            rows.append({"setup": name, "field": FIELD_LABELS.get(field, field), "invoices": total,
                         "correct": s["correct"], "wrong": s["wrong"], "missing": s["missing"],
                         "accuracy (= recall)": acc, "precision": prec, "F1": f1})
    return rows


def pooled(scores):
    c, w, m = (sum(s[k] for s in scores.values()) for k in ("correct", "wrong", "missing"))
    return rates(c, w, m)


def show_scores(table):
    display(table.style.format({c: PCT for c in ["accuracy (= recall)", "precision", "F1"]}, na_rep="–")
            .hide(axis="index"))


def bars(table, value, title, ax):
    table.pivot(index="field", columns="setup", values=value).plot.barh(ax=ax, xlim=(0, 1))
    ax.set(title=title, xlabel=value, ylabel="")
    ax.xaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
    ax.legend(loc="lower left", fontsize=8)


print("repository:", ROOT)
print("app python:", APP_PY.exists(), "| training python:", TRAIN_PY.exists())

## 1. Ground truth without manual labelling

Most Slovak invoices carry a **PAY by square** QR code, Czech ones a **QR Platba**
code. The issuer's accounting system generates them from the invoice data and
the extractor never reads them, so they are an independent answer key for the
**amount to pay, variable symbol and due date**. The archive file name adds the
**direction** (received invoices are filed under the internal number assigned on
approval) and, for issued invoices, the **invoice number**.

Safeguards: PAY by square payloads are CRC-checked; documents with two different
payment codes, or an issued invoice whose QR variable symbol differs from its
file name, get no labels from that source.

In [ ]:
if RUN["labels"]:
    run(APP_PY, "evaluation.payment_qr", "--pdfs", DOCS, "--out", LABELS)

labels = read_csv(LABELS)
if labels is not None:
    columns = {"typ": "Direction", "cislo_faktury": "Invoice number", "suma_s_dph": "Amount to pay",
               "variabilny_symbol": "Variable symbol", "datum_splatnosti": "Due date"}
    coverage = pd.DataFrame([{"field": name, "documents with a known value": int((labels[c] != "").sum())}
                             for c, name in columns.items()])
    print(f"{len(labels)} labelled documents")
    display(coverage.style.hide(axis="index"))
    display(labels["typ"].replace({"Prijatá": "received", "Vystavená": "issued", "": "unknown"})
            .value_counts().rename("documents").to_frame())

## 2. Benchmark: rules vs rules + Gemma 4

The production extractor on every document: the PDF text layer or PaddleOCR
(PP-OCRv6, GPU) for scans → rules with arithmetic validation → for received
invoices with critical fields still empty, Gemma 4 (local, via Ollama) fills the
gaps. *Accuracy (= recall)*: share of invoices with the right value; a wrong and an
empty value both count as misses. *Precision*: share of filled-in values that are
right — the one that matters most, because a wrong value can slip past review.

In [ ]:
if RUN["benchmark"]:
    run(APP_PY, "evaluation", "--pdfs", DOCS, "--labels", LABELS, "--llm", LLM, "--out", BENCHMARK)

summary = load(BENCHMARK / "summary.json")
if summary:
    print(f"{summary['documents']} documents: {summary['text_layer_docs']} digital, {summary['ocr_docs']} scanned")
    bench = pd.DataFrame([r for s in summary["setups"] if "scores" in s for r in score_rows(s["name"], s["scores"])])
    show_scores(bench)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    bars(bench, "accuracy (= recall)", "Accuracy per field", axes[0])
    bars(bench, "precision", "Precision per field", axes[1])
    plt.tight_layout()
    plt.show()
    for s in summary["setups"]:
        if "scores" in s:
            acc, prec, f1 = pooled(s["scores"])
            print(f"{s['name']}: all fields pooled - accuracy {acc:.1%}, precision {prec:.1%}, F1 {f1:.1%}; "
                  f"every field right on {s['perfect']} of {s['scored_docs']} invoices")

In [ ]:
if summary:
    speed = pd.DataFrame([{"stage": label, **t} for label, t in summary["timings"] if t])
    display(speed.style.format({c: "{:.2f} s" for c in ["median", "mean", "p90", "max"]}).hide(axis="index"))

## 3. Supplier split

A model that has seen a supplier's layout in training would look better on that
supplier's test invoices than on a new supplier's. Documents are therefore grouped
by counterparty first — the supplier's IBAN from the QR code for received
invoices, else the IČO or name — and whole groups go to train (70 %), validation
(10 %) or test (20 %) by a stable hash.

In [ ]:
if RUN["split"]:
    run(APP_PY, "training.split", "--labels", LABELS, "--predictions", BENCHMARK / "per_document.csv",
        "--out", SPLIT)

split = read_csv(SPLIT)
if split is not None:
    table = split.assign(kind=split["typ"].replace({"Prijatá": "received", "Vystavená": "issued", "": "unknown"}))
    counts = table.pivot_table(index="split", columns="kind", values="file", aggfunc="count", fill_value=0)
    counts["supplier groups"] = table.groupby("split")["group"].nunique()
    display(counts.loc[[s for s in ["train", "val", "test"] if s in counts.index]])

## 4. Own OCR: a CRNN text recognizer trained from scratch

No pretrained weights. A CNN reads the line image, a two-layer bidirectional LSTM
reads the features left to right, CTC loss aligns them with the text
(~8 M parameters). Training lines are **synthetic** — invoice field patterns
rendered with 30 system fonts and degraded like a scan — mixed with **real lines
cut from digital invoices**, whose text layer gives an exact transcription.
Below: synthetic samples (generated text, not invoice data).

In [ ]:
import random

from training.ocr import synth

fonts = synth.available_fonts()
sampler = synth.TextSampler()
fig, axes = plt.subplots(8, 1, figsize=(10, 6))
for i, ax in enumerate(axes):
    rng = random.Random(1000 + i)
    text = sampler.sample(rng)
    ax.imshow(synth.render_line(text, rng.choice(fonts), rng), cmap="gray", vmin=0, vmax=255)
    ax.set_title(text, fontsize=9, loc="left")
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
if RUN["real_lines"]:
    run(TRAIN_PY, "training.ocr.real_lines", "--pdfs", DOCS, "--split", SPLIT, "--out", REAL_LINES)
lines = read_csv(REAL_LINES / "lines.csv")
if lines is not None:
    display(lines.groupby("split").agg(lines=("text", "size"), documents=("document", "nunique")))

In [ ]:
if RUN["train_ocr"]:
    run(TRAIN_PY, "training.ocr.train", "--real", REAL_LINES, "--out", OCR_RUN)

history = load(OCR_RUN / "history.json")
if history:
    h = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for column in [c for c in h.columns if c.endswith("_cer")]:
        axes[0].plot(h["step"], h[column] * 100, marker="o", label=column.removesuffix("_cer"))
    axes[0].set(title="Character error rate on validation lines (lower is better)", xlabel="step", ylabel="CER %")
    axes[0].legend()
    axes[1].plot(h["step"], h["loss"], marker="o")
    axes[1].set(title="Training loss (CTC)", xlabel="step")
    plt.tight_layout()
    plt.show()
    display(h.tail(3).style.hide(axis="index"))

In [ ]:
if RUN["test_ocr"]:
    run(TRAIN_PY, "training.ocr.train", "--real", REAL_LINES, "--test", OCR_RUN / "best.pt")
ocr_test = load(OCR_RUN / "test_metrics.json")
if ocr_test:
    display(pd.DataFrame({name: {"CER": m["cer"], "line accuracy": m["line_accuracy"], "lines": m["lines"]}
                          for name, m in ocr_test.items() if isinstance(m, dict)}).T
            .style.format({"CER": PCT, "line accuracy": PCT}))

## 5. Own extraction model: LiLT fine-tuned on QR-labelled invoices

**LiLT** (`nielsr/lilt-xlm-roberta-base`, MIT licence) reads every word together
with its position on the page. Words come from the PDF text layer or PaddleOCR
word boxes — exactly what the app reads. Training labels are automatic: words that
read as the QR amount, variable symbol or due date are tagged (BIO scheme);
documents where a QR value was not found on the page are left out of training.
The test split is then compared with the rule-based extractor on the same invoices.

In [ ]:
if RUN["dump_words"]:
    run(APP_PY, "training.extraction.dump_words", "--pdfs", DOCS, "--out", WORDS)
if RUN["train_lilt"]:
    run(TRAIN_PY, "training.extraction.train_lilt", "--words", WORDS, "--labels", LABELS, "--split", SPLIT,
        "--out", LILT_RUN)

config = load(LILT_RUN / "config.json")
if config:
    print({k: config[k] for k in ["train_documents", "train_documents_skipped_unmatched", "val_documents",
                                  "train_chunks", "epochs", "lr"]})
history = load(LILT_RUN / "history.json")
if history:
    h = pd.DataFrame(history)
    fig, ax = plt.subplots(figsize=(8, 4))
    for column in [c for c in h.columns if c.startswith("val_acc")]:
        ax.plot(h["epoch"], h[column], marker="o", label=column.removeprefix("val_acc_"))
    ax.set(title="Validation accuracy per epoch", xlabel="epoch", ylim=(0, 1))
    ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}")
    ax.legend()
    plt.show()

In [ ]:
if RUN["test_lilt"]:
    run(TRAIN_PY, "training.extraction.train_lilt", "--words", WORDS, "--labels", LABELS, "--split", SPLIT,
        "--test", LILT_RUN / "best", "--baseline", BENCHMARK / "per_document.csv")

comparison = load(LILT_RUN / "test_summary.json")
if comparison:
    for group, setups in comparison.items():
        title = {"all": "All test invoices", "Prijatá": "Received", "Vystavená": "Issued"}.get(group, group)
        table = pd.DataFrame([r for name, s in setups.items() for r in score_rows(name, s["fields"])])
        print(f"\n{title}: {next(iter(setups.values()))['documents']} invoices from suppliers not seen in training")
        show_scores(table)
        fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
        bars(table, "accuracy (= recall)", f"{title}: accuracy", axes[0])
        bars(table, "precision", f"{title}: precision", axes[1])
        plt.tight_layout()
        plt.show()

## 6. Headline numbers

One table to quote: every scored field pooled. Benchmark rows cover all labelled
documents; model rows cover the test split only (suppliers never seen in training).

In [ ]:
rows = []
if summary:
    for s in summary["setups"]:
        if "scores" in s:
            acc, prec, f1 = pooled(s["scores"])
            rows.append({"what": s["name"], "documents": s["scored_docs"], "accuracy": acc, "precision": prec,
                         "F1": f1, "set": "all labelled documents"})
if comparison and "all" in comparison:
    for name, s in comparison["all"].items():
        acc, prec, f1 = pooled(s["fields"])
        rows.append({"what": name, "documents": s["documents"], "accuracy": acc, "precision": prec, "F1": f1,
                     "set": "test split"})
if rows:
    display(pd.DataFrame(rows).style.format({c: PCT for c in ["accuracy", "precision", "F1"]}, na_rep="–")
            .hide(axis="index"))
if ocr_test:
    for name, m in ocr_test.items():
        if isinstance(m, dict):
            print(f"Own OCR, {name.replace('_', ' ')}: character error rate {m['cer']:.2%} on {m['lines']} lines")